In [28]:
from netgen.occ import *
from netgen.meshing import IdentificationType
from ngsolve import *
from ngsolve.webgui import Draw

# 1. Geometry Setup
L = 2.0
pml_thickness = 1.0
z_source = 1.0  # Height of our invisible source plane

# Stack the boxes from bottom to top and name their materials
# 1. PML 
box_pml_d = Box((0, 0, -pml_thickness), (L/4, L/64, 0))
box_pml_d.mat("pml_region_down")  # <-- Added material name

box_pml_u = Box((0, 0, L), (L/4, L/64, L+pml_thickness))
box_pml_u.mat("pml_region_up")  # <-- Added material name

# 2. Lower Air (Between PML and Source)
box_air_bottom = Box((0, 0, 0), (L/4, L/64, z_source))
box_air_bottom.mat("air")  # <-- Added material name

# 3. Upper Air (Between Source and Rigid Wall)
box_air_top = Box((0, 0, z_source), (L/4, L/64, L))
box_air_top.mat("air")     # <-- Added material name

# Glue them into a single continuous domain
geometry = Glue([box_pml_d, box_air_bottom, box_air_top, box_pml_u])

# 2. Name the Faces
# Netgen's OCC allows us to gracefully select faces by their coordinate planes
geometry.faces[Z == z_source].name = "source_plane"

# Tag the boundaries for Boundary Conditions
geometry.faces.Max(Z).name = "top"
geometry.faces.Min(Z).name = "bottom"

# 3. Identify Periodic Faces (on the outer boundaries of the glued geometry)
geometry.faces.Max(X).Identify(geometry.faces.Min(X), "periodic_x")
geometry.faces.Max(Y).Identify(geometry.faces.Min(Y), "periodic_y")

# 4. Generate Mesh
geo = OCCGeometry(geometry)
mesh = Mesh(geo.GenerateMesh(maxh=0.05))

print("Mesh generated successfully! Notice the internal face at z=0.2")
Draw(mesh)

Mesh generated successfully! Notice the internal face at z=0.2


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [29]:
import cmath, math
# 1. Physics Parameters
freq = 343.0     
c_0 = 343.0* (1 + 1j * 0.0)  # Complex speed of sound to introduce a small amount of damping
k = 2 * math.pi * freq / c_0  

c_real = c_0.real 
m = 2                               # Quadratic profile
R0 = 1e-6                           # Target reflection coefficient (0.01%)
sigma0 = ((m + 1) * c_real / (2 * pml_thickness)) * math.log(1 / R0)
print(f"Calculated sigma0 for PML: {sigma0:.2f}")

# Incident angles (e.g., 45 degrees)
theta = (math.pi / 10)  # Polar angle (0 for normal incidence)
phi = 0.0

# Wave vector components
kx = k * math.sin(theta) * math.cos(phi)
ky = k * math.sin(theta) * math.sin(phi)
kz = k * math.cos(theta)

# Transverse wave vector for the Floquet shift
k_vec = CF((kx, ky, 0)) 

# Coordinate functions
x, y, z = x, y, z
print(f"Coordinate functions: x={x}, y={y}, z={z}")

# Calculate distance into the PML (0 inside the physical domain)
dist_top = IfPos(z - L, z - L, 0)
dist_bot = IfPos(-z, -z, 0)
dist = dist_top + dist_bot

# Quadratic damping profile
# sigma = sigma0 * (dist / pml_thickness)**2
sigma = sigma0 * (2* (dist / pml_thickness)**2 - (dist / pml_thickness)**4)

# Complex stretching factor
sz = 1 + 1j * (sigma / (2 * math.pi * freq))

# The anisotropic stretching matrix S = diag(sz, sz, 1/sz)
S = CF((1, 0, 0, 
        0, 1, 0, 
        0, 0, 1/sz**2), dims=(3,3))


# 2. Finite Element Space
fes = (Periodic(H1(mesh, order=3, complex=True)))

u, v = fes.TnT()

print(f"Degrees of freedom: {fes.ndof}")


Calculated sigma0 for PML: 7108.08
Coordinate functions: x=coef coordinate x, real
, y=coef coordinate y, real
, z=coef coordinate z, real

Degrees of freedom: 34030


In [30]:
from ngsolve import pml


# 2. Modified Gradients (Floquet Trick)
grad_u = grad(u) + 1j * k_vec * u
grad_v = grad(v) - 1j * k_vec * v 

# 3. Bilinear Form (LHS)
a = BilinearForm(fes)
a += (S* grad_u * grad_v - k**2 * u * v) * sz* dx

# 4. Linear Form (RHS) - The Transparent Source
f = LinearForm(fes)
source_func = exp(-40 * ((z-z_source)**2))  # A Gaussian source centered at (L/4, L/4, z_source)
f += source_func* v * sz* dx

# 5. Assemble and Solve
gfu = GridFunction(fes)

with TaskManager():
    a.Assemble()
    f.Assemble()
    gfu.vec.data = a.mat.Inverse(fes.FreeDofs()) * f.vec
    
print("Solve complete!")


Solve complete!


In [31]:
# 1. Reconstruct Field
phase = exp(1j * (kx * x + ky * y))
p_true = gfu * phase

# 3. Visualize
print("Rendering total pressure field (PML zeroed out)...")
Draw(p_true, mesh, "Acoustic Pressure", animate_complex=True)

Rendering total pressure field (PML zeroed out)...


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

BaseWebGuiScene